#Total revenue

How much revenue did the company generate?

In [0]:
%sql

SELECT
    ROUND(SUM(sales_amount), 2) AS total_revenue
FROM e2e_project.gold.fact_sales;

#Total orders



In [0]:
%sql

SELECT
    COUNT(DISTINCT order_number) AS total_orders
FROM e2e_project.gold.fact_sales;

COUNT(*)                 = number of sales lines
COUNT(DISTINCT order_number) = number of orders

#Total customers who purchased

This counts customers who actually appear in sales.

In [0]:
%sql

SELECT
    COUNT(DISTINCT customer_key) AS active_customers
FROM e2e_project.gold.fact_sales;

# Average Order Value

Average Order Value

=
Total Revenue / Number of Orders

In [0]:
%sql

SELECT
    ROUND(
        SUM(sales_amount)
        /
        COUNT(DISTINCT order_number),
        2
    ) AS average_order_value
FROM e2e_project.gold.fact_sales;

# Revenue by country

In [0]:
%sql

SELECT
    c.country,
    ROUND(SUM(f.sales_amount), 2) AS total_revenue
FROM e2e_project.gold.fact_sales f
LEFT JOIN e2e_project.gold.dim_customer c
    ON f.customer_key = c.customer_key
GROUP BY c.country
ORDER BY total_revenue DESC;

Revenue is a measure from the fact.

Country is an attribute from a dimension.

# Revenue by category

In [0]:
%sql

SELECT
    p.category,
    ROUND(SUM(f.sales_amount), 2) AS total_revenue
FROM e2e_project.gold.fact_sales f
LEFT JOIN e2e_project.gold.dim_product p
    ON f.product_key = p.product_key
GROUP BY p.category
ORDER BY total_revenue DESC;

# Revenue by subcategory

In [0]:
%sql

SELECT
    p.category,
    p.subcategory,
    ROUND(SUM(f.sales_amount), 2) AS total_revenue
FROM e2e_project.gold.fact_sales f
LEFT JOIN e2e_project.gold.dim_product p
    ON f.product_key = p.product_key
GROUP BY
    p.category,
    p.subcategory
ORDER BY total_revenue DESC;

# Top 10 products

In [0]:
%sql

SELECT
    p.product_name,
    p.category,
    ROUND(SUM(f.sales_amount), 2) AS total_revenue,
    SUM(f.quantity) AS units_sold
FROM e2e_project.gold.fact_sales f
LEFT JOIN e2e_project.gold.dim_product p
    ON f.product_key = p.product_key
GROUP BY
    p.product_name,
    p.category
ORDER BY total_revenue DESC
LIMIT 10;

# Top 10 customers

Who are our most valuable customers?

In [0]:
%sql

SELECT
    c.customer_number,
    CONCAT(
        COALESCE(c.first_name, ''),
        ' ',
        COALESCE(c.last_name, '')
    ) AS customer_name,
    c.country,
    ROUND(SUM(f.sales_amount), 2) AS total_revenue,
    COUNT(DISTINCT f.order_number) AS total_orders
FROM e2e_project.gold.fact_sales f
LEFT JOIN e2e_project.gold.dim_customer c
    ON f.customer_key = c.customer_key
GROUP BY
    c.customer_number,
    c.first_name,
    c.last_name,
    c.country
ORDER BY total_revenue DESC
LIMIT 10;

# Monthly sales trend

In [0]:
%sql

SELECT
    DATE_TRUNC('month', order_date) AS sales_month,
    ROUND(SUM(sales_amount), 2) AS revenue
FROM e2e_project.gold.fact_sales
WHERE order_date IS NOT NULL
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY sales_month;

# Yearly sales

In [0]:
%sql

SELECT
    YEAR(order_date) AS sales_year,
    ROUND(SUM(sales_amount), 2) AS revenue
FROM e2e_project.gold.fact_sales
WHERE order_date IS NOT NULL
GROUP BY YEAR(order_date)
ORDER BY sales_year;

# Now learn a useful advanced KPI: YoY growth : Year-over-Year Growth

In [0]:
%sql

WITH yearly_sales AS (
    SELECT
        YEAR(order_date) AS sales_year,
        SUM(sales_amount) AS revenue
    FROM e2e_project.gold.fact_sales
    WHERE order_date IS NOT NULL
    GROUP BY YEAR(order_date)
),

with_previous AS (
    SELECT
        sales_year,
        revenue,
        LAG(revenue) OVER (
            ORDER BY sales_year
        ) AS previous_year_revenue
    FROM yearly_sales
)

SELECT
    sales_year,
    ROUND(revenue, 2) AS revenue,
    ROUND(previous_year_revenue, 2) AS previous_year_revenue,

    ROUND(
        (
            revenue - previous_year_revenue
        )
        / previous_year_revenue
        * 100,
        2
    ) AS yoy_growth_percent

FROM with_previous
ORDER BY sales_year;

# Revenue and profit are NOT the same

Profit

=
sales amount - cost amount

In [0]:
%sql

SELECT
    ROUND(SUM(f.sales_amount), 2) AS revenue,

    ROUND(
        SUM(p.cost * f.quantity),
        2
    ) AS estimated_cost,

    ROUND(
        SUM(
            f.sales_amount -
            (p.cost * f.quantity)
        ),
        2
    ) AS estimated_profit

FROM e2e_project.gold.fact_sales f

LEFT JOIN e2e_project.gold.dim_product p
    ON f.product_key = p.product_key;

I would call this estimated profit, not accounting profit, because our dataset does not necessarily contain every real company expense.

# Profit by category

In [0]:
%sql

SELECT
    p.category,

    ROUND(
        SUM(f.sales_amount),
        2
    ) AS revenue,

    ROUND(
        SUM(p.cost * f.quantity),
        2
    ) AS estimated_cost,

    ROUND(
        SUM(
            f.sales_amount -
            (p.cost * f.quantity)
        ),
        2
    ) AS estimated_profit

FROM e2e_project.gold.fact_sales f

LEFT JOIN e2e_project.gold.dim_product p
    ON f.product_key = p.product_key

GROUP BY p.category

ORDER BY estimated_profit DESC;

![image_1788862638377.png](./image_1788862638377.png "image_1788862638377.png")

# Gold quality summary.

In [0]:
%sql

SELECT
    COUNT(*) AS fact_rows,
    COUNT(DISTINCT order_number) AS orders,
    COUNT(DISTINCT customer_key) AS customers,
    COUNT(DISTINCT product_key) AS products,
    SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END)
        AS missing_customer_keys,
    SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END)
        AS missing_product_keys
FROM e2e_project.gold.fact_sales;

![image_1788862709498.png](./image_1788862709498.png "image_1788862709498.png")

Databricks SQL dashboard from these queries: KPI cards, monthly revenue trend, revenue by category, top products, and customers.